Augmentation inspired from: https://www.kaggle.com/code/ayushdabra/inceptionresnetv2-unet-81-dice-coeff-86-acc#Data-Augmentation-using-Albumentations-Library
# Import data from Kaggle into Drive

In [ ]:
from google.colab import files
files.upload()
!rm -r ~/.kaggle
!mkdir ~/.kaggle
!mv ./kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json
rm: cannot remove '/root/.kaggle': No such file or directory


In [ ]:
!kaggle datasets download -d bulentsiyah/semantic-drone-dataset

100% 3.87G/3.89G [00:42<00:00, 98.3MB/s]
100% 3.89G/3.89G [00:42<00:00, 98.9MB/s]


## Option 1: Unzip the folder to 1 drive

In [ ]:
'''import os
# Put into one drive
!cd ~
zip_path = '/content/drive/MyDrive/0 cnn training/data/2 semantic-aerial/semantic-drone-dataset.zip'
os.getcwd()
!cp '{zip_path}'
!unzip -q 'semantic-drone-dataset.zip'
os.listdir()
'''

"import os\n# Put into one drive\n!cd ~\nzip_path = '/content/drive/MyDrive/0 cnn training/data/2 semantic-aerial/semantic-drone-dataset.zip'\nos.getcwd()\n!cp '{zip_path}'\n!unzip -q 'semantic-drone-dataset.zip'\nos.listdir()\n"

## Option 2: Just use notebook

In [ ]:
!unzip -q '/content/semantic-drone-dataset.zip'
!rm '/content/semantic-drone-dataset.zip'

# Import libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import cv2
import os

import albumentations as A

from sklearn.model_selection import train_test_split

# Dataset understanding and splitting

In [2]:
# 1. Import data by path
image_path = '/content/dataset/semantic_drone_dataset/original_images/'
mask_path = '/content/dataset/semantic_drone_dataset/label_images_semantic/'

In [4]:
# 2. Class
class_df = pd.read_csv('/content/class_dict_seg.csv', index_col=False, skipinitialspace=True)
class_df

,name,r,g,b
0,unlabeled,0,0,0
1,paved-area,128,64,128
2,dirt,130,76,0
3,grass,0,102,0
4,gravel,112,103,87
5,water,28,42,168
6,rocks,48,41,30
7,pool,0,50,89
8,vegetation,107,142,35
9,roof,70,70,70


In [17]:
num_classes = len(class_df) -1 # -1 for header
print('Number of classes: ', num_classes)
files = os.listdir(image_path)
print('Number of images: ', len(files))

Number of classes:  23
Number of images:  400


## Notes:
1. There are 23 classes
2. The raw images consistently consist of (4000,6000,3)
3. There are no test/train/validate

Ideas:
1. Split data set into 6:2:2, train:val:test
2. Augment data with Albumentations
3. Build U-net model
4. Train and build model

# Data augmentation with albumentations

The following techniques will be used:
1. rotate
2. horizontal flip
3. vertical flip
4. random brightness and contrast
5. shift scale rotate
6. grid distortion
7. optical distortion
8. contrast limited adaptive histogram equalization (clahe)

In [ ]:
def augment(width, height):
  transform = A.compose([
      A.Rotate(p=1.0,interpolation=cv2.INTER_NEAREST),
      A.HorizontalFlip(p=1.0),
      A.VerticalFlip(p=1.0),
      A.RandomBrightnessContrast(p=1.0),
      A.ShiftScaleRotate(p=1.0),
      A.GridDistortion(p=0.5),
      A.OpticalDistortion(p=0.5,interpolation=cv2.INTER_NEAREST),
      A.CLAHE(p=0.5)
  ])
  return transform